> ## SOLUTION / ANSWER KEY &mdash; Lab 8.10
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-10-observability.ipynb`](../lab-10-observability.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.10 &mdash; Observability for a Team

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Log every agent action into an auditable trace
- Count calls per agent to find the busy/faulty one
- Detect a handoff loop so a runaway team is caught

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; routing, synthesis, tool wiring, agent structure &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Failure modes & observability](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-08-10"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
A multi-agent system fails in new ways &mdash; **handoff loops**, agents **talking past** each other,
**cost blow-ups**, and **lost accountability** (deck slide 19). The defence is **observability**: log
every agent, message, handoff and decision so you can **replay** the conversation, find the **faulty
agent**, and watch **cost**. **LangSmith / Langfuse** do this for graphs; here you build the offline
version.

In [ ]:
from collections import Counter
print("we log (agent, action, detail) events and watch for loops")

## Your Turn
Complete the `AgentTrace` (log + read back) and `detect_loop` (a runaway handoff).

In [ ]:
from collections import Counter

class AgentTrace:
    def __init__(self):
        self.events = []
    def log(self, agent, action, detail):
        self.events.append((agent, action, detail))
    def agents_involved(self):
        return [a for a, _, _ in self.events]
    def calls_per_agent(self):
        return Counter(a for a, _, _ in self.events)

def detect_loop(path, limit=2):
    # a runaway loop: some agent appears MORE than `limit` times (a normal 2x back-and-forth is fine)
    return any(c > limit for c in Counter(path).values())

try:
    tr = AgentTrace()
    tr.log('supervisor', 'route', 'billing+tech')
    tr.log('billing', 'tool', 'lookup_invoice')
    tr.log('tech', 'tool', 'known_issues')
    print('involved :', tr.agents_involved())
    print('calls    :', dict(tr.calls_per_agent()))
    print('loop?    :', detect_loop(['a', 'b', 'a', 'b', 'a', 'b']))
    print('healthy? :', detect_loop(['supervisor', 'billing', 'tech']))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("log records an event", lambda: (lambda t: (t.log("a", "x", "y"), len(t.events))[1])(AgentTrace()) == 1)
expect_true("agents_involved lists them in order", lambda: (lambda t: [t.log(a, "x", "y") for a in ("supervisor", "billing", "tech")] and t.agents_involved())(AgentTrace()) == ["supervisor", "billing", "tech"])
expect_true("calls_per_agent counts each agent", lambda: (lambda t: [t.log("billing", "x", "y") for _ in range(3)] and t.calls_per_agent()["billing"])(AgentTrace()) == 3)
expect_true("detect_loop catches a runaway handoff", lambda: detect_loop(["a", "b", "a", "b", "a", "b"]) is True)
expect_true("detect_loop clears a healthy path", lambda: detect_loop(["supervisor", "billing", "tech"]) is False)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; the real LangChain interface (not graded)
See the REAL callback / tracing interface LangChain exposes (LangSmith / Langfuse capture full graphs). It needs only `pip install langchain-core` (already in the course env) and makes **no** network
call. **The graded steps above never call an LLM, so the lab always verifies offline.**

In [ ]:
try:
    from langchain_core.callbacks import BaseCallbackHandler
    class AgentHandler(BaseCallbackHandler):
        def on_chain_end(self, outputs, **kw):
            print("agent/chain finished ->", outputs)
    print("Real LangChain/LangGraph call handlers like AgentHandler on every agent & tool event.")
    print("For full multi-agent run traces: set LANGCHAIN_TRACING_V2=true + LANGCHAIN_API_KEY (LangSmith),")
    print("or run Langfuse (this course's stack) and attach its callback handler to the graph.")
except Exception as e:
    print("Install langchain-core to use real callbacks -- skipping:", type(e).__name__)
print("The AgentTrace above already logged every agent, action and detail offline.")

---
### Key takeaway
Log every agent, message, handoff and decision so you can replay the conversation, find the faulty agent and watch cost. A multi-agent system is only as trustworthy as it is observable.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>